# WEEK 5

## RAG - Retrival Augmented Generation

We've already used techniques to improve prompts:
- multi-shot prompting
- tools
- additional context

We can take this to the next level building a database of expert information, called a 'Knowledge Base'

When a user asks a question, the we can search for anything relevant in the Knowledge base, and add relevant details to the prompt sent to the model.



### A small example

Business set up:
- a small insuarance tech start up
- we have a knowledge base of the company shared drive
- task is to build an AI knowledge Worker

Toy implementation:
- read in names of products and employees
- see if a question refers to employee or products by name
- add relevant details to the prompt



# Encoding LLMs and Vector Embeddings

#### Auto-encoding vs Auto-regressive llms
- Auto-regressive LLMs predict future token from the past
- Auto-encoding LLMs produce output based on the full input

#### Auto-Encoding LLMs
- Applications include sentiment analysis and classification
- used to calculate "Vector Embeddings" representing an input as a list of numbers ie. a vector
- Examples include google's BERT and OpenAIEmbeddings

Vector embeddings
- can represent a character, token, word, document or something abstract.
- hundreds or thousands of dimension
- Represent an 'understanding' of the input. related concepts should be closer in dimensional space than less related concepts.
- support vector math - eg. Vec(King) - Vec(Man) + Vec(Woman) = Vec(Queen).


# The BIG idea behind RAG

- Encoding LLM - takes text and tokenizes it
- Vector Datastore

Sequence:
- User Question
- Encoding LLM tokenizes/vectorizes the question
- Retrieve the "nearby" vectors of the question from the Vector Datastore
- Add that context to the prompt that goes to the LLM.


## Day 2

# LangChain
- Opensource framework created in 2022
- Provides a common framework for interfacing with many LLMs
- Includes its own declarative language: LangChain Expression Language (LCEL)

- Greatly simplifies creation of applications using LLMs
- Wrapper code arround LLMs make it easy to swap models
- As APIs for LLMs have matured, converged and simplified, the need for a unifying framework has decreased.



## Using LangChain to load our knowledgebase

- Read in documents
- Add meta-data to the documents
- Break down the contents into overlapping chunks

## Day 3

# Vector Datastore

Vector embedding models

word2Vec (2013) - deep neural network that converts words to vectors that retains "meaning"

BERT (2018) - Transformer model for encoding

OpenAI Embeddings (2024) - What we'll use

## Introducing Chroma
A vector datastore. 

Creating the chroma datastore and populating vector embeddings was only 2 lines of code!

embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embeddings=embeddings,
    persist_directiory=db_name
)

## Day 4

# Implementing a full RAG pipeline

#### Key abstractions in LangChain
- LLM, represents an LLM
- Retriever, interface to the Vectorstore (RAG Retrieval)
- Memory, History of queries.


In [ ]:
# create a new chat with openAI
llm = ChatOpenAI(temperature=0.7)

#conversation memory
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

#create a retriever from Chroma datastore
retriever = vectorstore.as_retriever()

#Putting them together
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

#### FAISS - Facebook AI Similarity Search

Not really a vector datastore, but a library to search for vectors near other vectors. Doesnt persist on disk, but easy to drop in place of Chroma.

In [ ]:

# BEFORE
# vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)

# AFTER
vectorstore = FAISS.from_documents(chunks, embedding=embeddings)

## Some differences in calling functions
total_vectors = vectorstore.index.ntotal
dimensions = vectorstore.index.d


# Prework for Plotly
vectors = []
documents = []
doc_types = []
colors = []
color_map = {'products':'blue', 'employees':'green', 'contracts':'red', 'company':'orange'}

for i in range(total_vectors):
    vectors.append(vectorstore.index.reconstruct(i))
    doc_id = vectorstore.index_to_docstore_id[i]
    document = vectorstore.docstore.search(doc_id)
    documents.append(document.page_content)
    doc_type = document.metadata['doc_type']
    doc_types.append(doc_type)
    colors.append(color_map[doc_type])
    
vectors = np.array(vectors)

## Day 5

#### LCEL - LangChain Expression Language
- maps closely to the python workflow for constructing pipelines.
- Takes the form of a YAML file


# Behind the Curtain

- use callbacks to see the actual prompt sent to the model.

- Use callbacks to diagnose and fix common problems: Sending the wrong chunks to the model.

#### Solutions

Revising chunking:
- Send in whole documents as chunk
- Create more chunks
- Increase chunk overlap
- Control the number of chunks sent to the model


In [ ]:
# the retriever is an abstraction over the VectorStore that will be used during RAG; k is how many chunks to use
retriever = vectorstore.as_retriever(search_kwargs={"k": 25})

#### Sending MORE chunks to LLMs tends to work well.

- LLMs are good at filtering out irrelevant context
- Excetp for "Chain of Thought" models, additional context can slow them or confuse.

# Week 5 Exercise

## Create a knowledge Worker on your information to boost productivity

- Assmble your files in 1 place; a personal Knowledge Base
- Vectorize everything in Chroma - your vector datastore
- Build a conversational AI to ask questions!

##### Advanced ideas to take to the next level
- If you use Google Workspace, use Google API to read your docs
- If you use MS Office, use libraries to read office docs
- Harder - use libraries to connect your email inbox, slack etc.

In [1]:
# imports

import os
import glob
from dotenv import load_dotenv
import gradio

In [26]:
# langchain

from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.embeddings import HuggingFaceEmbeddings

from langchain_core.callbacks import StdOutCallbackHandler


In [19]:
# Model names
MODEL = "gpt-4o-mini"
db_name = "exercise_vector_db"

In [4]:
# Load environment variables in a file called .env

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

In [ ]:
# Reading documents from file

directory = "C:/Users/cbenn/OneDrive/Documents/Game Design/Torchlight/Class Designs/*"

folders = glob.glob(directory)

def add_metadata(doc, doc_type):
    doc.metadata["doc_type"] = doc_type
    return doc

text_loader_kwargs = {'encoding': 'utf-8'}

documents = []
for folder in folders:
    print(folder)
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()
    documents.extend([add_metadata(doc, doc_type) for doc in folder_docs])

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Total number of chunks: {len(chunks)}")
print(f"Document types found: {set(doc.metadata['doc_type'] for doc in documents)}")

In [20]:
## Vector Embedding

embeddings = OpenAIEmbeddings()

# Clear previous database files
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

# Create a new vectorstore

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")


Vectorstore created with 96 documents


In [38]:
## For now ignore Deprecation warning!!
# create a new chat with openAI
llm = ChatOpenAI(temperature=0.7, model=MODEL)

#conversation memory
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

#create a retriever from Chroma datastore
retriever = vectorstore.as_retriever(search_kwargs={"k":5})

#Putting them together
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory, callbacks=[StdOutCallbackHandler()])

In [ ]:
query = "What is a powerful fire damage skill for the Corsair?"
result = conversation_chain.invoke({"question": query})
print(result["answer"])

In [ ]:
# Including Gradio

def chat(message, history):
    result = conversation_chain.invoke({"question":message})
    return result["answer"]



gradio.ChatInterface(chat, type="messages").launch(inbrowser=True)